# acai.xml_parser — Test & Examples Notebook

Interactive test suite for the hexagonal XML parser module.
Run each cell top-to-bottom. A temp directory is used for all file I/O and cleaned up at the end.

**Architecture under test:**

| Layer | Component | Description |
|-------|-----------|-------------|
| Port | `XmlParserPort` | Abstract `parse(path) → ParsedLawDocument` interface |
| Config | `XmlParserConfig` | AKN namespace URI, XML namespace, language |
| Domain | `ParsedLawDocument`, `LawMetadata`, `ParsedArticle` | Result models |
| Adapter | `LxmlParser` | Concrete lxml-based AKN parser |
| Factory | `create_xml_parser()` | Composition root |

## 0 — Setup & Imports

In [1]:
import sys, shutil, textwrap, tempfile
from pathlib import Path
from dataclasses import asdict

# Ensure acai is importable
_project_root = Path.cwd()
while _project_root.name != "acai" and _project_root != _project_root.parent:
    _project_root = _project_root.parent
_shared_python = _project_root.parent  # …/shared/python
if str(_shared_python) not in sys.path:
    sys.path.insert(0, str(_shared_python))

from acai.logging import create_logger, LoggerConfig, LogLevel
from acai.xml_parser import (
    create_xml_parser,
    XmlParserPort,
    XmlParserConfig,
    XmlParseError,
    ConfigurationError,
    ParsedLawDocument,
    LawMetadata,
    ParsedArticle,
)
from acai.xml_parser.adapters.outbound.lxml_parser import LxmlParser

# Shared logger
_logger = create_logger(LoggerConfig(service_name="xml-parser-test", log_level=LogLevel.WARNING))

# Shared temp directory — cleaned up in the last cell
WORK_DIR = Path(tempfile.mkdtemp(prefix="acai_xmlparser_test_"))
print(f"Working directory: {WORK_DIR}")

# AKN namespace constant
AKN_NS = "http://docs.oasis-open.org/legaldocml/ns/akn/3.0"

# Test result tracker
_results: list[tuple[str, bool, str]] = []

def _record(name: str, passed: bool, detail: str = "") -> None:
    _results.append((name, passed, detail))
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}  {name}" + (f"  — {detail}" if detail else ""))

print("Imports OK ✔")

Working directory: C:\Users\micha\AppData\Local\Temp\acai_xmlparser_test_zh4p_5j1
Imports OK ✔


## 1 — Sample XML Fixtures

Create AKN XML files in the temp directory for all subsequent tests.

In [2]:
MINIMAL_XML = textwrap.dedent(f"""\
    <?xml version="1.0" encoding="UTF-8"?>
    <akomaNtoso xmlns="{AKN_NS}">
      <act name="publicLaw">
        <meta>
          <identification source="#ch.bk">
            <FRBRWork>
              <FRBRnumber value="210"/>
              <FRBRname xml:lang="de" value="Schweizerisches Zivilgesetzbuch" shortForm="ZGB"/>
              <FRBRname xml:lang="fr" value="Code civil suisse" shortForm="CC"/>
            </FRBRWork>
          </identification>
        </meta>
        <body>
          <section eId="sec_1">
            <num>Erster Titel</num>
            <heading>Einleitung</heading>
            <article eId="art_1">
              <num>Art. 1</num>
              <paragraph eId="art_1__para_1">
                <content><p>Das Gesetz findet auf alle Rechtsfragen Anwendung.</p></content>
              </paragraph>
            </article>
            <article eId="art_2">
              <num>Art. 2</num>
              <paragraph eId="art_2__para_1">
                <content><p>Jedermann hat nach Treu und Glauben zu handeln.</p></content>
              </paragraph>
              <paragraph eId="art_2__para_2">
                <content><p>Missbrauch eines Rechtes findet keinen Rechtsschutz.</p></content>
              </paragraph>
            </article>
          </section>
        </body>
      </act>
    </akomaNtoso>
""")

EMPTY_BODY_XML = textwrap.dedent(f"""\
    <?xml version="1.0" encoding="UTF-8"?>
    <akomaNtoso xmlns="{AKN_NS}">
      <act name="publicLaw">
        <meta><identification source="#ch.bk">
          <FRBRWork><FRBRnumber value="999"/></FRBRWork>
        </identification></meta>
        <body/>
      </act>
    </akomaNtoso>
""")

PREAMBLE_XML = textwrap.dedent(f"""\
    <?xml version="1.0" encoding="UTF-8"?>
    <akomaNtoso xmlns="{AKN_NS}">
      <act name="publicLaw">
        <meta><identification source="#ch.bk">
          <FRBRWork><FRBRnumber value="101"/></FRBRWork>
        </identification></meta>
        <preamble><p>Die Bundesversammlung beschliesst:</p></preamble>
      </act>
    </akomaNtoso>
""")

NESTED_XML = textwrap.dedent(f"""\
    <?xml version="1.0" encoding="UTF-8"?>
    <akomaNtoso xmlns="{AKN_NS}">
      <act name="publicLaw">
        <meta><identification source="#ch.bk">
          <FRBRWork>
            <FRBRnumber value="220"/>
            <FRBRname xml:lang="de" value="Obligationenrecht" shortForm="OR"/>
          </FRBRWork>
        </identification></meta>
        <body>
          <part eId="part_1">
            <num>Erster Teil</num>
            <heading>Allgemeine Bestimmungen</heading>
            <title eId="tit_1">
              <num>Erster Titel</num>
              <heading>Die Entstehung der Obligationen</heading>
              <chapter eId="chap_1">
                <num>Erstes Kapitel</num>
                <heading>Die Entstehung durch Vertrag</heading>
                <article eId="art_1">
                  <num>Art. 1</num>
                  <paragraph eId="art_1__para_1">
                    <content><p>Willenseinigung erforderlich.</p></content>
                  </paragraph>
                </article>
              </chapter>
            </title>
          </part>
        </body>
      </act>
    </akomaNtoso>
""")

# Write all fixtures to disk
def _write(name: str, content: str) -> Path:
    p = WORK_DIR / name
    p.write_text(content, encoding="utf-8")
    return p

minimal_path = _write("minimal.xml", MINIMAL_XML)
empty_path   = _write("empty_body.xml", EMPTY_BODY_XML)
preamble_path = _write("preamble.xml", PREAMBLE_XML)
nested_path  = _write("nested.xml", NESTED_XML)

print(f"Created 4 fixture files in {WORK_DIR}")

Created 4 fixture files in C:\Users\micha\AppData\Local\Temp\acai_xmlparser_test_zh4p_5j1


## 2 — Factory: `create_xml_parser()`

Create an `XmlParserPort` via the factory with default and custom configs.

In [3]:
# Default factory → LxmlParser
parser = create_xml_parser(_logger)
_record("Factory — returns LxmlParser", isinstance(parser, LxmlParser))
_record("Factory — satisfies XmlParserPort", isinstance(parser, XmlParserPort))

# Custom config (French)
fr_parser = create_xml_parser(_logger, XmlParserConfig(language="fr"))
_record("Factory — custom config accepted", isinstance(fr_parser, LxmlParser))

✅ PASS  Factory — returns LxmlParser
✅ PASS  Factory — satisfies XmlParserPort
✅ PASS  Factory — custom config accepted


## 3 — Metadata Extraction

Parse `FRBRnumber`, `FRBRname`, and `shortForm` from the AKN meta section.

In [4]:
result = parser.parse(minimal_path)

_record("Metadata — law_number", result.metadata.law_number == "210", f"got '{result.metadata.law_number}'")
_record("Metadata — law_name (de)", result.metadata.law_name == "Schweizerisches Zivilgesetzbuch",
        f"got '{result.metadata.law_name}'")
_record("Metadata — short_form", result.metadata.short_form == "ZGB", f"got '{result.metadata.short_form}'")
_record("Metadata — is LawMetadata", isinstance(result.metadata, LawMetadata))

# French config extracts French name
fr_result = fr_parser.parse(minimal_path)
_record("Metadata — law_name (fr)", fr_result.metadata.law_name == "Code civil suisse",
        f"got '{fr_result.metadata.law_name}'")
_record("Metadata — short_form (fr)", fr_result.metadata.short_form == "CC",
        f"got '{fr_result.metadata.short_form}'")

✅ PASS  Metadata — law_number  — got '210'
✅ PASS  Metadata — law_name (de)  — got 'Schweizerisches Zivilgesetzbuch'
✅ PASS  Metadata — short_form  — got 'ZGB'
✅ PASS  Metadata — is LawMetadata
✅ PASS  Metadata — law_name (fr)  — got 'Code civil suisse'
✅ PASS  Metadata — short_form (fr)  — got 'CC'


## 4 — Article Extraction

Parse articles from the `<body>`, verifying article numbers, paragraph text, and counts.

In [5]:
articles = result.articles

_record("Articles — found 2", len(articles) == 2, f"got {len(articles)}")
_record("Articles — all ParsedArticle", all(isinstance(a, ParsedArticle) for a in articles))
_record("Articles — Art. 1 number", articles[0].article == "Art. 1")
_record("Articles — Art. 2 number", articles[1].article == "Art. 2")

# Paragraph content
_record("Articles — Art. 1 has paragraph", any("Anwendung" in p for p in articles[0].paragraphs))
_record("Articles — Art. 2 has 2 paragraphs", len(articles[1].paragraphs) >= 2,
        f"got {len(articles[1].paragraphs)}")

# Level is integer
_record("Articles — level is int", all(isinstance(a.level, int) for a in articles))

# Print for visual inspection
for a in articles:
    print(f"\n  {a.article} (level={a.level}, headings={a.headings})")
    for i, p in enumerate(a.paragraphs, 1):
        print(f"    {i}. {p[:80]}")

✅ PASS  Articles — found 2  — got 2
✅ PASS  Articles — all ParsedArticle
✅ PASS  Articles — Art. 1 number
✅ PASS  Articles — Art. 2 number
✅ PASS  Articles — Art. 1 has paragraph
✅ PASS  Articles — Art. 2 has 2 paragraphs  — got 2
✅ PASS  Articles — level is int

  Art. 1 (level=1, headings=['Erster Titel Einleitung'])
    1. Das Gesetz findet auf alle Rechtsfragen Anwendung.

  Art. 2 (level=1, headings=['Erster Titel Einleitung'])
    1. Jedermann hat nach Treu und Glauben zu handeln.
    2. Missbrauch eines Rechtes findet keinen Rechtsschutz.


## 5 — Heading Hierarchy

Verify that headings are collected from article up to the root, ordered root → leaf.

In [6]:
nested_result = parser.parse(nested_path)
art = nested_result.articles[0]

_record("Headings — found for nested article", len(art.headings) >= 2, f"got {art.headings}")
_record("Headings — contains 'Allgemeine Bestimmungen'",
        any("Allgemeine Bestimmungen" in h for h in art.headings))
_record("Headings — contains 'Vertrag'",
        any("Vertrag" in h for h in art.headings))

# Order: root appears before leaf
combined = " ".join(art.headings)
allg_pos = combined.find("Allgemeine Bestimmungen")
vertrag_pos = combined.find("Vertrag")
_record("Headings — root before leaf", allg_pos < vertrag_pos,
        f"positions: Allgemeine={allg_pos}, Vertrag={vertrag_pos}")

print(f"\nHeadings for {art.article}: {art.headings}")

✅ PASS  Headings — found for nested article  — got ['Erster Teil Allgemeine Bestimmungen', 'Erster Titel Die Entstehung der Obligationen', 'Erstes Kapitel Die Entstehung durch Vertrag']
✅ PASS  Headings — contains 'Allgemeine Bestimmungen'
✅ PASS  Headings — contains 'Vertrag'
✅ PASS  Headings — root before leaf  — positions: Allgemeine=12, Vertrag=117

Headings for Art. 1: ['Erster Teil Allgemeine Bestimmungen', 'Erster Titel Die Entstehung der Obligationen', 'Erstes Kapitel Die Entstehung durch Vertrag']


## 6 — Edge Cases

Empty body returns no articles. Preamble-only documents use the preamble as fallback.

In [7]:
# Empty body → no articles
empty_result = parser.parse(empty_path)
_record("Edge — empty body returns []", empty_result.articles == [])
_record("Edge — empty body metadata present", empty_result.metadata.law_number == "999")

# Preamble fallback → at least one article
pre_result = parser.parse(preamble_path)
_record("Edge — preamble fallback returns articles", len(pre_result.articles) >= 1,
        f"got {len(pre_result.articles)}")
_record("Edge — preamble content captured",
        any("Bundesversammlung" in p for a in pre_result.articles for p in a.paragraphs))

2026-03-30 19:56:00,757 - WARNING - No articles found in empty_body.xml


✅ PASS  Edge — empty body returns []
✅ PASS  Edge — empty body metadata present
✅ PASS  Edge — preamble fallback returns articles  — got 1
✅ PASS  Edge — preamble content captured


## 7 — Error Handling

Missing files, invalid XML, and bad config raise the expected exceptions.

In [8]:
# Missing file → XmlParseError
caught_missing = False
try:
    parser.parse(WORK_DIR / "nonexistent.xml")
except XmlParseError:
    caught_missing = True
_record("Error — missing file raises XmlParseError", caught_missing)

# Invalid XML → XmlParseError
bad_path = WORK_DIR / "bad.xml"
bad_path.write_text("this is not xml {{{", encoding="utf-8")
caught_bad = False
try:
    parser.parse(bad_path)
except XmlParseError:
    caught_bad = True
_record("Error — invalid XML raises XmlParseError", caught_bad)

# Empty namespace → ConfigurationError
caught_config = False
try:
    XmlParserConfig(akn_namespace_uri="")
except ConfigurationError:
    caught_config = True
_record("Error — empty namespace raises ConfigurationError", caught_config)

✅ PASS  Error — missing file raises XmlParseError
✅ PASS  Error — invalid XML raises XmlParseError
✅ PASS  Error — empty namespace raises ConfigurationError


## 8 — Context Manager

In [9]:
with create_xml_parser(_logger) as ctx_parser:
    ctx_result = ctx_parser.parse(minimal_path)
    _record("Context manager — parse inside block", len(ctx_result.articles) == 2)

_record("Context manager — exit succeeded", True)

✅ PASS  Context manager — parse inside block
✅ PASS  Context manager — exit succeeded


## 9 — Real Fedlex File (optional)

If XML files from the data pipeline exist, parse one to verify against real-world AKN documents.

In [10]:
real_dir = Path(__file__).resolve().parents[5] / "data" / "20-xml-store" if "__file__" in dir() else Path.cwd().parents[2] / "data" / "20-xml-store"

# Also check common workspace locations
for candidate in [real_dir, Path("data/20-xml-store"), Path("../../data/20-xml-store")]:
    if candidate.exists():
        real_dir = candidate
        break

if real_dir.exists():
    real_files = sorted(real_dir.glob("*.xml"))[:1]
    if real_files:
        real_result = parser.parse(real_files[0])
        _record("Real file — parsed without error", isinstance(real_result, ParsedLawDocument))
        _record("Real file — has metadata", real_result.metadata.law_number is not None,
                f"SR {real_result.metadata.law_number}")
        _record("Real file — has articles", len(real_result.articles) > 0,
                f"{len(real_result.articles)} articles")
        print(f"\n  File: {real_files[0].name}")
        print(f"  Law:  SR {real_result.metadata.law_number} — {real_result.metadata.law_name}")
        print(f"  Articles: {len(real_result.articles)}")
    else:
        print("No XML files found in data directory.")
else:
    print(f"Data directory not found (checked {real_dir}). Skipping real-file test.")

Data directory not found (checked d:\Development\2026 LexChat\acai-powertools\lib\data\20-xml-store). Skipping real-file test.


## Cleanup

In [11]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f"Removed temp directory: {WORK_DIR}")
print(f"Directory exists after cleanup: {WORK_DIR.exists()}")

Removed temp directory: C:\Users\micha\AppData\Local\Temp\acai_xmlparser_test_zh4p_5j1
Directory exists after cleanup: False


## Summary

In [12]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total  = len(_results)

print(f"\n{'='*60}")
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print(f"{'='*60}\n")

for name, ok, detail in _results:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}" + (f"  — {detail}" if detail else ""))


  RESULTS: 29/29 passed, 0 failed

  [PASS] Factory — returns LxmlParser
  [PASS] Factory — satisfies XmlParserPort
  [PASS] Factory — custom config accepted
  [PASS] Metadata — law_number  — got '210'
  [PASS] Metadata — law_name (de)  — got 'Schweizerisches Zivilgesetzbuch'
  [PASS] Metadata — short_form  — got 'ZGB'
  [PASS] Metadata — is LawMetadata
  [PASS] Metadata — law_name (fr)  — got 'Code civil suisse'
  [PASS] Metadata — short_form (fr)  — got 'CC'
  [PASS] Articles — found 2  — got 2
  [PASS] Articles — all ParsedArticle
  [PASS] Articles — Art. 1 number
  [PASS] Articles — Art. 2 number
  [PASS] Articles — Art. 1 has paragraph
  [PASS] Articles — Art. 2 has 2 paragraphs  — got 2
  [PASS] Articles — level is int
  [PASS] Headings — found for nested article  — got ['Erster Teil Allgemeine Bestimmungen', 'Erster Titel Die Entstehung der Obligationen', 'Erstes Kapitel Die Entstehung durch Vertrag']
  [PASS] Headings — contains 'Allgemeine Bestimmungen'
  [PASS] Headings — co